# AirBnB NYC Analytics Project

* Project by Nhi Bui · Villanova University · [GitHub](https://github.com/nhibui23/airbnb-nyc-product-analytics-project) · [LinkedIn](https://linkedin.com/in/nhiuyenbui)

## 03. Guest Decision Drivers

> "When a guest is choosing between listings, which features actually influence their decision?"

Notebook 02 showed that the 3 Q1 guest-side features, price tier, Instant Book, and host verification, all have small or zero visible gaps in average rating. This notebook tests whether those gaps are real signals or sampling noise.

**My Approach:**
* Instant Book and host verification are binary (yes/no), so each gets a Welch's t-test
* Price tier has 4 groups (Low, Medium, High, Very High), so it gets a one-way ANOVA with Tukey's HSD post-hoc test

**Outcome variable:** `review rate number` (rating proxy for guest satisfaction)

**Threshold:** p < 0.05 for statistical significance

In [4]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/Airbnb_Open_Data_Cleaned.csv')

# Apply Airbnb branding
airbnb_coral = '#FF5A5F'
airbnb_teal = '#00A699'
airbnb_orange = '#FC642D'
airbnb_dark = '#484848'
airbnb_gray = '#767676'

airbnb_palette = [airbnb_coral, airbnb_teal, airbnb_orange, airbnb_dark, airbnb_gray]
sns.set_palette(airbnb_palette)
sns.set_style('whitegrid')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.family'] = 'DejaVu Sans'

print(f"Dataset: {df.shape[0]:,} listings")

Dataset: 63,718 listings


# Test 1: Instant Book

* Instant Book lets guests reserve without host approval. 

* The EDA showed nearly identical averages for both groups (~3.3 stars)

* We will conduct a Welch's t-test to check whether the difference is statistically zero

In [5]:
# Split by Instant Book status
ib_true = df[df['instant_bookable'] == True]['review rate number']
ib_false = df[df['instant_bookable'] == False]['review rate number']

In [6]:
# Welch's t-test
t_stat, p_value = stats.ttest_ind(ib_true, ib_false, equal_var=False)

print(f"Instant Book = True:  mean = {ib_true.mean():.4f}, n = {len(ib_true):,}")
print(f"Instant Book = False: mean = {ib_false.mean():.4f}, n = {len(ib_false):,}")
print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

Instant Book = True:  mean = 3.2962, n = 31,576
Instant Book = False: mean = 3.2850, n = 32,142

T-statistic: 1.1017
P-value: 0.2706


### Key Takeaway
 
* p = 0.27 > 0.05. The difference is not statistically significant.

→ **Observation:** Instant Book has no measurable effect on guest ratings. The feature may serve a convenience role in the booking flow, but it doesn't influence how guests rate their stay.

# Test 2: Host Verification

Host verification confirms a host's identity. The EDA showed a 0.01-star gap between verified and unconfirmed hosts. SImilar to Instant Book, we will conduct Welch's t-test to check whether this gap is real.

In [8]:
# Split by verification status
verified = df[df['host_identity_verified'] == 'verified']['review rate number']
unconfirmed = df[df['host_identity_verified'] == 'unconfirmed']['review rate number']


## Post-hoc test

To find out which specific room types differ from each other, you need a post-hoc test using Tukey's HSD 

In [9]:

# Welch's t-test
t_stat, p_value = stats.ttest_ind(verified, unconfirmed, equal_var=False)

print(f"Verified:    mean = {verified.mean():.4f}, n = {len(verified):,}")
print(f"Unconfirmed: mean = {unconfirmed.mean():.4f}, n = {len(unconfirmed):,}")
print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

Verified:    mean = 3.2911, n = 31,854
Unconfirmed: mean = 3.2900, n = 31,864

T-statistic: 0.1113
P-value: 0.9114


### Key Takeaway

* p = 0.9114 > 0.05. The difference is not statistically significant.

→ **Observation:** Same as Instant Bookable, host verification has no measurable effect on guest ratings. Verification belongs in trust and safety, not in the quality-improvement signals of guests

# Test 3: Price Tier

The EDA showed that Low-priced listings underperform, while Medium, High, and Very High cluster together. 

Instead of conducting t-test for binary values, we will conduct a one-way ANOVA to test whether the overall difference across groups is significant and Tukey's HSD to see which specific pairs differ.

In [10]:
# Recreate price bins
df['price_group'] = pd.cut(df['price'],
                           bins=[0, 50, 500, 1000, 1500],
                           labels=['Low', 'Medium', 'High', 'Very High'])

In [11]:
# Split groups
low = df[df['price_group'] == 'Low']['review rate number']
medium = df[df['price_group'] == 'Medium']['review rate number']
high = df[df['price_group'] == 'High']['review rate number']
very_high = df[df['price_group'] == 'Very High']['review rate number']

In [12]:
# ANOVA
f_stat, p_value = stats.f_oneway(low, medium, high, very_high)
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")


F-statistic: 5.5889
P-value: 0.0008


In [13]:
# Sample sizes
print(f"\nSample sizes:")
print(df['price_group'].value_counts())


Sample sizes:
price_group
High         27684
Medium       24823
Very High    11144
Low             67
Name: count, dtype: int64


In [14]:
# Tukey's HSD post-hoc test
tukey = pairwise_tukeyhsd(df['review rate number'], df['price_group'], alpha=0.05)
print(tukey)

  Multiple Comparison of Means - Tukey HSD, FWER=0.05  
group1   group2  meandiff p-adj   lower   upper  reject
-------------------------------------------------------
  High       Low  -0.1429 0.7972 -0.5446  0.2587  False
  High    Medium   0.0153 0.5159 -0.0134   0.044  False
  High Very High  -0.0427 0.0153 -0.0796 -0.0059   True
   Low    Medium   0.1583 0.7422 -0.2434    0.56  False
   Low Very High   0.1002 0.9191 -0.3022  0.5026  False
Medium Very High  -0.0581 0.0004 -0.0955 -0.0206   True
-------------------------------------------------------


### Key Takeaway

* ANOVA p = 0.0008 < 0.05. The overall difference across price tiers is statistically significant.

**Tukey's HSD identifies 2 significant pairs:**
* High vs Very High (p = 0.0153)
* Medium vs Very High (p = 0.0004)

→ Both pairs involve Very High priced listings rating lower than the mid-range tiers.

→ **Observation:** The significant effect is driven by Very High priced listings underperforming. Budget (Low) vs everything else is not significant, likely due to the small sample size in the Low tier (67 listings).

→ **Note:** The Low tier's small sample means its budget finding is not conclusive and not negative, so having a larger sample might reveal its real gap.

## Summary

| Feature | Test | p-value | Significant? | Finding |
|---|---|---|---|---|
| Instant Book | Welch's t-test | [VALUE] | No | No effect on rating |
| Host verification | Welch's t-test | 0.9114 | No | No effect on rating |
| Price tier (overall) | One-way ANOVA | 0.0008 | Yes | Significant difference across tiers |
| 1. High vs Very High | Tukey HSD | 0.0153 | Yes | Very High underperforms |
| 2. Medium vs Very High | Tukey HSD | 0.0004 | Yes | Very High underperforms |

→ **2 of 3 Q1 features show no statistical effect on guest ratings.** The only confirmed signal is price

* Very High priced listings underperform mid-range tiers

* Instant Book and host verification can therefore be deprioritized as rating drivers

# Handoff

These findings feed into 2 downstream notebooks:

**Notebook 04 - Host Revenue Model:** The price tier finding here matters for hosts setting prices. The availability ANOVA will show whether limited-availability hosts outperform on both rating and occupancy.

**Notebook 06 - Final Recommendation:** All 3 test results are combined into the final recommendation set. Specific product recommendations will be made in this final notebook